In [ ]:
# Run directory setup (Drive-friendly)
from datetime import datetime
from pathlib import Path
import os
import yaml

# User-editable: base folder on Drive
BASE_ARTIFACTS = "/content/drive/MyDrive/thermal_rl"
SURROGATE = "rf"
POLICY = "sac"
SHIELD = "shielded"
SEED = 42

run_id = datetime.now().strftime("run_%Y-%m-%d_%H-%M") + f"_seed{SEED}"
RUN_DIR = f"{BASE_ARTIFACTS}/{SURROGATE}/{POLICY}/{SHIELD}/{run_id}"
print("RUN_DIR:", RUN_DIR)
os.makedirs(RUN_DIR, exist_ok=True)

# Create a run-specific SAC config that writes everything into RUN_DIR
PROJECT_ROOT = Path("/content/thermal-adt")
SRC_CFG = PROJECT_ROOT / "configs/rl/sac_shielded.yaml"
DST_CFG = PROJECT_ROOT / "configs/rl/sac_drive.yaml"

cfg = yaml.safe_load(SRC_CFG.read_text())
cfg["save_dir"] = RUN_DIR

# Optional: also set rf_model_path to a clean path in this repo
# (default in sac_shielded.yaml may still point to an old location)
# cfg["rf_model_path"] = "models/rf_teacher.pkl"

DST_CFG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote run-specific config:", DST_CFG)
print("save_dir set to:", RUN_DIR)

In [ ]:
# Standalone evaluation + export artifacts to RUN_DIR/eval_standalone
from pathlib import Path
import os, json, yaml, numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Load config to find SAVE_DIR
CFG_PATH = "/content/Digital-twin1/configs/rl_training.yaml"
with open(CFG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

SAVE_DIR = Path(cfg["save_dir"]).expanduser()
MODEL_PATH = SAVE_DIR / "best_model" / "best_model.zip"
if not MODEL_PATH.exists():
    MODEL_PATH = SAVE_DIR / "final_model.zip"
VECNORM_PATH = SAVE_DIR / "vec_normalize.pkl"
EVAL_DIR = SAVE_DIR / "eval_standalone"
PLOTS_DIR = EVAL_DIR / "plots"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

N_EVAL_EPISODES = int(cfg.get("n_eval_episodes", 5))

# Build eval environment (VecEnv API)
from rl.env_thermal_rf import make_thermal_env
from rl.safety_shield import SafetyWrapper
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

env_cfg = cfg.get("environment", {})
safety_cfg = cfg.get("safety", {})
use_safety = cfg.get("use_safety_shield", True)

rf_model_path = Path(cfg["rf_model_path"]).expanduser()

def _make_env():
    env = make_thermal_env(
        rf_model_path=rf_model_path,
        config=env_cfg,
        workload_profile="stress",
    )
    env = Monitor(env)
    if use_safety:
        env = SafetyWrapper(env, safety_cfg)
    return env

vec_base = DummyVecEnv([_make_env])
if VECNORM_PATH.exists():
    eval_env = VecNormalize.load(str(VECNORM_PATH), vec_base)
    eval_env.training = False
    eval_env.norm_obs = True
    eval_env.norm_reward = False
else:
    eval_env = vec_base

# Load model
from stable_baselines3 import SAC
agent = SAC.load(str(MODEL_PATH), env=eval_env, device="auto")
print(f"[INFO] Loaded model for eval: {MODEL_PATH}")

# Run episodes and collect metrics
rows = []
for ep in range(N_EVAL_EPISODES):
    obs = eval_env.reset()
    done = False
    ep_rew = 0.0
    steps = 0
    while not done:
        action, _ = agent.predict(obs, deterministic=True)
        obs, r, d, info = eval_env.step(action)
        ep_rew += float(r[0]) if isinstance(r, (list, np.ndarray)) else float(r)
        done = bool(d[0]) if isinstance(d, (list, np.ndarray)) else bool(d)
        steps += 1

    # Try to fetch episode metrics from base env
    metrics = {}
    base_vec = getattr(eval_env, "venv", eval_env)
    maybe_env = getattr(base_vec, "envs", [base_vec])[0]
    env_ref = maybe_env
    for _ in range(6):
        getter = getattr(env_ref, "get_episode_metrics", None)
        if callable(getter):
            metrics = getter() or {}
            break
        env_ref = getattr(env_ref, "env", None) or getattr(env_ref, "unwrapped", None)
        if env_ref is None:
            break

    row = {
        "episode": ep + 1,
        "total_reward": ep_rew,
        "steps": steps,
        "throttle_events": metrics.get("throttle_events", 0),
        "max_temp": metrics.get("max_temp", np.nan),
        "min_headroom": metrics.get("min_headroom", np.nan),
        "energy_proxy": metrics.get("total_energy", np.nan),
    }
    rows.append(row)
    print(f"Episode {ep+1}: Reward={row['total_reward']:.2f}, Throttles={row['throttle_events']}")

# Save per-episode CSV
episodes_df = pd.DataFrame(rows)
episodes_csv = EVAL_DIR / "episodes.csv"
episodes_df.to_csv(episodes_csv, index=False)
print(f"[INFO] Wrote {episodes_csv}")

# Summary stats
summary = episodes_df.agg({
    "total_reward": ["mean", "std"],
    "throttle_events": ["mean", "std"],
    "max_temp": ["mean", "std"],
    "min_headroom": ["mean", "std"],
    "energy_proxy": ["mean", "std"],
}).T
summary_csv = EVAL_DIR / "summary.csv"
summary.to_csv(summary_csv)
print(f"[INFO] Wrote {summary_csv}")

summary_json = {
    k: {
        "mean": float(v["mean"]) if not np.isnan(v["mean"]) else None,
        "std": float(v["std"]) if not np.isnan(v["std"]) else None,
    }
    for k, v in summary.to_dict()["mean"].items()
}
with open(EVAL_DIR / "summary.json", "w") as f:
    json.dump(summary_json, f, indent=2)

# Simple plots
plt.figure(figsize=(6,4))
plt.bar(["Reward"], [episodes_df["total_reward"].mean()], yerr=[episodes_df["total_reward"].std() or 0.0])
plt.title("Episodic Reward (mean ± std)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "reward_mean.png", dpi=200)
plt.close()

plt.figure(figsize=(6,4))
plt.bar(["Throttle Events"], [episodes_df["throttle_events"].mean()])
plt.title("Throttle Events (mean)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "throttles_mean.png", dpi=200)
plt.close()

print(f"[INFO] Plots saved under {PLOTS_DIR}")


# RL Training (SAC) using RF Surrogate

This notebook trains the SAC agent using the Random Forest surrogate dynamics model **in the reorganized `thermal-adt` repository**.

## Steps
1. Upload `thermal-adt.zip` to Colab
2. Run cells top-to-bottom

## 1) GPU runtime (recommended)
Runtime -> Change runtime type -> GPU

In [ ]:
!nvidia-smi

## 2) Upload + Extract Project Zip
Upload `thermal-adt.zip` into `/content/` using the Colab file browser.

In [ ]:
!rm -rf /content/thermal-adt
!unzip -q /content/thermal-adt.zip -d /content/thermal-adt
!ls -la /content/thermal-adt | head

## 3) Mount Google Drive (recommended for checkpointing)
This prevents losing progress if Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3b) Point `save_dir` to Google Drive
This ensures checkpoints and logs persist across runtime disconnects.

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('configs/rl_training.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# Save all RL artifacts to Google Drive
drive_save_dir = '/content/drive/MyDrive/rl_artifacts/sac_thermal'
cfg['save_dir'] = drive_save_dir

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Updated save_dir to:', drive_save_dir)
print('Config file:', cfg_path)

## 3) Install RL dependencies

In [ ]:
%cd /content/thermal-adt
!pip -q install -r requirements.txt

## 4) Add project root to `PYTHONPATH`

In [ ]:
import sys
sys.path.insert(0, '/content/thermal-adt')
print(sys.path[0])

## 5) Sanity check: RF teacher bundle exists

In [ ]:
from pathlib import Path
import yaml

# Check RF bundle path from config
CFG = Path('/content/thermal-adt/configs/rl/sac_drive.yaml')
cfg = yaml.safe_load(CFG.read_text())
rf_path = Path('/content/thermal-adt') / cfg.get('rf_model_path', 'models/rf_teacher.pkl')
print('Config rf_model_path:', cfg.get('rf_model_path'))
print('Resolved RF path:', rf_path)
print('Exists:', rf_path.exists())

if not rf_path.exists():
    print('If this is False, either:')
    print('  1) Train RF in Colab using scripts/training/train_rf.py and output bundle to models/, or')
    print('  2) Upload/copy your existing rf_teacher.pkl into /content/thermal-adt/models/')

## Resume training after a disconnect
If Colab disconnects, re-run from the top (mount Drive, unzip, install), then resume using `--resume`.

Checkpoints are saved under your Drive `save_dir/checkpoints/`.

In [ ]:
# Resume training after a disconnect
# If Colab disconnects, re-run from the top (mount Drive, unzip, install), then resume.

!python scripts/training/train_sac.py --config configs/rl/sac_drive.yaml --resume

## 6) (Optional) Quick smoke test
Run a short training to verify everything works before the full 1M timesteps.

In [ ]:
## 6) Quick smoke test (100 timesteps)
This validates that training runs end-to-end before you launch a long run.

import yaml
from pathlib import Path

base_cfg = Path('configs/rl/sac_drive.yaml')
cfg = yaml.safe_load(base_cfg.read_text())

# Override training settings for smoke test
cfg.setdefault('training', {})
cfg['training']['total_timesteps'] = 100
cfg['training']['checkpoint_freq'] = 50
cfg['training']['eval_freq'] = 50
cfg['training']['n_eval_episodes'] = 2

smoke_cfg = Path('configs/rl/sac_smoke.yaml')
smoke_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Wrote:', smoke_cfg)

In [ ]:
!python scripts/training/train_sac.py --config configs/rl/sac_smoke.yaml

## 7) Full training run
This will train for 1,000,000 timesteps (edit `configs/rl_training.yaml` to change).

In [ ]:
## 7) Full training run
Edit `configs/rl/sac_drive.yaml` to set `training.total_timesteps` as needed (e.g., 200000).

!python scripts/training/train_sac.py --config configs/rl/sac_drive.yaml

## 8) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/Digital-twin1/rl_logs/tensorboard

## 9) Evaluate only
Loads the best model (if present) or final model and runs evaluation episodes.

In [ ]:
!python -m rl.training.train_sac --config configs/rl_training.yaml --device auto --eval-only